## Insider Threat Radar – Behavior Analytics for Employee Risk Prediction

### Feature Engineering 

#### Import Libraries

In [1]:
import pandas as pd
import numpy as np

#### Load Clean Data

In [2]:
email_df = pd.read_csv("../Dataset/email_clean.csv")
psycho_df = pd.read_csv("../Dataset/psychometric_clean.csv")

print("Clean datasets loaded ✅")

Clean datasets loaded ✅


#### Convert Date

In [3]:
email_df['date'] = pd.to_datetime(email_df['date'])

### 🔹 LOGIN / EMAIL Behavior Features

#### Emails Per User

In [4]:
email_count = email_df.groupby('user').size().reset_index(name='total_emails')
email_count.head()

,user,total_emails
0,AAE0190,4711
1,AAF0535,480
2,AAF0791,3012
3,AAL0706,336
4,AAM0658,659


#### Attachment Behavior

In [5]:
attach_features = email_df.groupby('user')['attachments'].sum().reset_index()
attach_features.rename(columns={'attachments':'total_attachments'}, inplace=True)
attach_features.head()

,user,total_attachments
0,AAE0190,1780
1,AAF0535,364
2,AAF0791,0
3,AAL0706,145
4,AAM0658,613


#### Email Size Feature

In [6]:
size_features = email_df.groupby('user')['size'].mean().reset_index()
size_features.rename(columns={'size':'avg_email_size'}, inplace=True)
size_features.head()

,user,avg_email_size
0,AAE0190,30020.394184
1,AAF0535,30397.402083
2,AAF0791,29958.497676
3,AAL0706,29828.181548
4,AAM0658,29895.532625


### 🔹 OFF-HOUR Access

#### Extract Hour

In [7]:
email_df['hour'] = email_df['date'].dt.hour

#### Off-hour Emails

**(Office hours: 9AM – 6PM)**

In [8]:
email_df['off_hour'] = email_df['hour'].apply(
    lambda x: 1 if (x < 9 or x > 18) else 0
)

off_hour_feat = email_df.groupby('user')['off_hour'].sum().reset_index()
off_hour_feat.rename(columns={'off_hour':'off_hour_emails'}, inplace=True)
off_hour_feat.head()

,user,off_hour_emails
0,AAE0190,499
1,AAF0535,1
2,AAF0791,377
3,AAL0706,54
4,AAM0658,146


### 🔹 TIME-BASED Features

#### Weekday vs Weekend

In [9]:
email_df['day'] = email_df['date'].dt.day_name()

email_df['is_weekend'] = email_df['day'].isin(['Saturday','Sunday']).astype(int)

weekend_feat = email_df.groupby('user')['is_weekend'].sum().reset_index()
weekend_feat.rename(columns={'is_weekend':'weekend_emails'}, inplace=True)
weekend_feat.head()

,user,weekend_emails
0,AAE0190,0
1,AAF0535,0
2,AAF0791,0
3,AAL0706,0
4,AAM0658,0


### 🔹 Merge All Features

#### Combine All

In [10]:
features = email_count.merge(attach_features, on='user') \
                       .merge(size_features, on='user') \
                       .merge(off_hour_feat, on='user') \
                       .merge(weekend_feat, on='user')

features.head()

,user,total_emails,total_attachments,avg_email_size,off_hour_emails,weekend_emails
0,AAE0190,4711,1780,30020.394184,499,0
1,AAF0535,480,364,30397.402083,1,0
2,AAF0791,3012,0,29958.497676,377,0
3,AAL0706,336,145,29828.181548,54,0
4,AAM0658,659,613,29895.532625,146,0


### 🔹 Add PSYCHOMETRIC Features

#### Merge Personality

In [11]:
final_features = pd.merge(
    features,
    psycho_df,
    left_on='user',
    right_on='user_id',
    how='left'
)

final_features.head()

,user,total_emails,total_attachments,avg_email_size,off_hour_emails,weekend_emails,employee_name,user_id,O,C,E,A,N
0,AAE0190,4711,1780,30020.394184,499,0,August Armando Evans,AAE0190,36,30,14,50,29
1,AAF0535,480,364,30397.402083,1,0,Athena Amelia Foreman,AAF0535,17,21,36,33,31
2,AAF0791,3012,0,29958.497676,377,0,Aladdin Abraham Foley,AAF0791,14,40,40,50,34
3,AAL0706,336,145,29828.181548,54,0,April Alika Levy,AAL0706,37,14,28,13,25
4,AAM0658,659,613,29895.532625,146,0,Abel Adam Morton,AAM0658,43,35,37,36,22


### 🔹 HANDLE Missing Personality

#### Fill NaN

In [12]:
trait_cols = ['O','C','E','A','N']
final_features[trait_cols] = final_features[trait_cols].fillna(
    final_features[trait_cols].mean()
)

#### Save FEATURE MATRIX

In [13]:
final_features.to_csv("../Dataset/final_features.csv", index=False)

print("Feature file saved ✅")

Feature file saved ✅
